In [23]:
#from datetime import datetime
from pathlib import Path
notebook_directory = Path.cwd()

#import numpy as np
import pandas as pd
import xlwings as xw

In [24]:
prices_path = (
    notebook_directory.parent.parent
    / "backtesting"
    / "historical prices"
    / "sectors.csv"
)
prices_df = pd.read_csv(prices_path, parse_dates=["date"])


db_path = (
    notebook_directory.parent
    / "spreadsheets"
    / "2026 Fin Inst Database.xlsx"
)
wb = xw.Book(db_path)
ws = wb.sheets["SECTOR ETFs"]
db_df = ws.tables["Table1"].range.options(pd.DataFrame, header=1, index=False).value


anchor_path = (
    notebook_directory.parent
    / "spreadsheets"
    / "2026 Group Trading Inputs.xlsm"
)
wb = xw.Book(anchor_path)
ws = wb.sheets["FI INPUTs"]
anchor_df = ws.tables["fi_inputs"].range.options(pd.DataFrame, header=1, index=False).value


In [25]:
# symbols = anchor_df['my_fi_name'].to_list()

symbols = ['FSTA', 'XLP', 'VDC']
anchor = 'VDC'


In [26]:
prices_df["date"] = pd.to_datetime(prices_df["date"], errors="coerce")

'''
previous_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).to_period("M")
prices_df = prices_df.loc[prices_df["date"].dt.to_period("M").eq(previous_month)].copy()
'''

#'''
start_date = pd.Timestamp("2024-01-02")
end_date = pd.Timestamp("2026-08-15")
#'''

prices_df = prices_df.loc[prices_df["date"].between(start_date, end_date)].copy()


In [27]:
'''
div_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).strftime("%Y%m")
div_column = f"Div {div_month}"
'''

div_column = "Div 202606"

In [28]:
for sym in symbols:
    dividends = db_df.loc[db_df["Symbol"].eq(sym), div_column].to_list()
    div_amt = float(dividends[0])

    prices_df[f"{sym}*"] = prices_df[sym] - div_amt

In [29]:
for sym in symbols:

    # anchor = anchor_df.loc[anchor_df["my_fi_name"].eq(sym), "anchor_fi"].to_list()[0]

    prices_df[f"{sym}**"] = prices_df[f'{anchor}*'] / prices_df[f'{sym}*']
    avg_ratio = prices_df[f"{sym}**"].mean()
    anchor_df.loc[anchor_df['my_fi_name'] == sym, "multiplier"] = avg_ratio

In [30]:
anchor_df = anchor_df.drop(columns="moving_avg_days")

In [31]:
print(anchor_df)

   my_fi_name my_pf_name anchor_fi  multiplier
0         AGG       IBKR       AGG         NaN
1         BND       IBKR       AGG         NaN
2        SCHZ       IBKR       AGG         NaN
3        SPAB       IBKR       AGG         NaN
4        ANGL       IBKR      ANGL         NaN
5        FALN       IBKR      ANGL         NaN
6        CORP       IBKR      CORP         NaN
7        SPBO       IBKR      CORP         NaN
8        USIG       IBKR      CORP         NaN
9         VTC       IBKR      CORP         NaN
10        EMB       IBKR       EMB         NaN
11       VWOB       IBKR       EMB         NaN
12       BNDX       IBKR      IAGG         NaN
13       IAGG       IBKR      IAGG         NaN
14        CWB       IBKR      ICVT         NaN
15       ICVT       IBKR      ICVT         NaN
16        BWX       IBKR      IGOV         NaN
17       IGOV       IBKR      IGOV         NaN
18        HYG       IBKR       JNK         NaN
19       HYLB       IBKR       JNK         NaN
20        JNK